# Corpus embedding — RUN ON GOOGLE COLAB (T4 GPU), ~5 minutes

Embeds the 52k-passage corpus with the two BERT-size models the laptop can't handle.

Steps:
1. Runtime → Change runtime type → **T4 GPU** → Save
2. Upload `chunks_c512o16.jsonl` (from `IEEE_Paper/data/`) via the folder icon
3. Runtime → Run all
4. `corpus_embeddings.zip` downloads automatically — unzip it into data/ for local index assembly

In [ ]:
import torch, os
assert torch.cuda.is_available(), 'Enable T4 GPU: Runtime -> Change runtime type'
assert os.path.exists('chunks_c512o16.jsonl'), 'Upload chunks_c512o16.jsonl first (folder icon, left sidebar)'
!pip install -q sentence-transformers
print('ready')

In [ ]:
import json, numpy as np
from sentence_transformers import SentenceTransformer

chunks = [json.loads(l) for l in open('chunks_c512o16.jsonl')]
texts = [c['text'] for c in chunks]
print(len(texts), 'chunks')

MODELS = {
    'bge-base-en-v1-5_c512o16': 'BAAI/bge-base-en-v1.5',
    'pubmedbert-base-embeddings_c512o16': 'NeuML/pubmedbert-base-embeddings',
}
outputs = []
for fp, model_id in MODELS.items():
    print('==', model_id)
    model = SentenceTransformer(model_id, device='cuda')
    emb = model.encode(texts, batch_size=256, normalize_embeddings=True,
                       show_progress_bar=True)
    out = f'emb_{fp}.npy'
    np.save(out, emb.astype(np.float16))
    outputs.append(out)
    del model
    torch.cuda.empty_cache()
print('done:', outputs)

In [ ]:
import zipfile
from google.colab import files
with zipfile.ZipFile('corpus_embeddings.zip', 'w') as z:
    for o in outputs:
        z.write(o)
files.download('corpus_embeddings.zip')